In [ ]:
# Lost in the Museum - reproduces the 0.86912 submission end to end
#
# Submit one fixed-length vector per image; the host cosine-matches each hidden
# query against the other 19,999 and scores Hit@3. No labels, so nothing is
# trained - every gain here is in how the descriptors are combined.
#
# The recipe, and why each part is there:
#
#   1. ViT-g/14 @518, CLS + GeM(p=3)              -> 3072 dims  (0.8285 alone)
#   2. ViT-L/14 @518, CLS + GeM, upright          -> 2048 dims  (0.79194 alone)
#   3. ViT-L/14 @518, CLS + GeM, rotated 180 deg  -> 2048 dims
#   4. Flip a candidate to its 180 view only when that view beats upright by
#      MARGIN. Many visitor photos are rotated by a multiple of 90 degrees and
#      DINOv2 has no invariance to it. Worth ~7 queries; flipping more
#      aggressively was measured and LOSES (margin 0.08 cost 0.0034).
#   5. L2-normalise each block SEPARATELY, then concatenate -> 5120 dims.
#      Without per-block L2 the larger block swamps the smaller one, which is
#      what pinned every earlier fusion attempt at 0.80201.
#   6. Whiten at FULL width - no PCA truncation.
#
# Step 6 matters most and is the easiest to get wrong. The leaderboard curve on
# this concat was 2048 -> 0.83557, 3072 -> 0.84563, 4096 -> 0.85234,
# 5120 (full) -> 0.86912: deltas +0.0101, +0.0067, then +0.0168. It ACCELERATED
# exactly where truncation stopped. At full dimension PCA is only an orthogonal
# rotation, which cosine ignores, so the transform degenerates to pure whitening
# and discards nothing. Truncating to 1536 costs 0.041.
import base64, glob, os, time, zlib
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
SIZE = 518
MARGIN = 0.15          # rotation flip gate; 0.08 loses 0.0034, 0.25 is identical
WORK = Path('/kaggle/working')
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

DATA = Path('/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset')
if not DATA.exists():
    hits = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
            if glob.glob(os.path.join(d, '*.png'))]
    assert hits, 'no PNG directory found under /kaggle/input'
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, '*.png')))))
paths = sorted(DATA.glob('*.png'))
names = np.array([p.name for p in paths])
assert len(paths) == 20000, f'expected 20000 images, got {len(paths)}'
print(f'{len(paths)} images from {DATA}')
print(torch.cuda.get_device_name(0))


In [ ]:
# Which images are eligible to be flipped.
#
# The 180-degree view is only worth considering for the ~1,000 visitor photos,
# not the 19,000 clean gallery scans - flipping a gallery scan can only hurt.
# Queries are identified by a deskew bounding-box test (scan_queries.py): the
# area gained by rotating an image to its tightest bounding box. On a random
# sample of 900 it flagged 5.0 percent, against a true query rate of exactly
# 1000/20000 = 5.0 percent.
#
# That scan is a slow deterministic CPU pass over 20,000 images, so its output -
# 3,439 indices into the sorted file list - is embedded here rather than
# recomputed. 6.5 KB, and it keeps this notebook self-contained.
CAND_B64 = (
    "eNol2meUTmfbBuBBRASJ3nsdvXejd4IIwiuiDBKM3nuJ3qKF0Rm9TIIgehstgyghSvReBqMz2ndkfT+O/9dz7fM+773XeuIE"
    "BAR8QnLSkYmsZCcvBSlKMcpQliAqUolafMX3tCWEzvSgJ30ZyBCG8hOjGcN0ZjKPRYTxK+v4nc38wVa2s5P9HOAQf3KLB0Tx"
    "jDe8IyBWQEAc4vElyUhOClKTluyUIIgq1KYODfiGprSiPT/ShT70YyDjmcBEpjCV6axgJb+zmW3sZC8RRHKME5zlAhe5zE1u"
    "EcVzYsUOCPiMBCQjHenJSFZykodCFKU4JalHSzrSnX4MYRQTmc1yNrCZbexmHxEcIpJjXOY6N7nFfZ7ynlhx7JkvSUpq0pCW"
    "dBSiMEFUoCLVqUEt6tCQ5rSkDR3oTHeGM5YZzGE+qwhnHVuI5AznucBFLnOD2zwgikc84TkveUNs4Y/DJyQgOanJSDbyUoyS"
    "lKEilahOTRrTlNa0oS3tCKEL3ehFb/rSj4EMYgQ/MYkpTGcRK1hNOOvZwGa2so0dRHKe29zlEdG85d1/vyduQEBCkpOJHOQk"
    "N3kpSBFKUY7K1KAWzehEF7rTiz4MYCCDGMJQhjGSUFawng1sZRf7iGA/f3KGq9zkPlE8IprYnwYEfMrnJCEZGchCEOWpQR3q"
    "Up+mtKANAxnHVGYyizCWs4KV/MYGNrOfk/zDeS7ykGhe8orXfOCTeAEBcYlPMgpQmIpUohrVqUN9GhBMCN3pSX8GMIjBjGAc"
    "M1lEGFvYRgSHOMJfnOQ8V7nOHe7ymnif2RlfkIQ0pCUjWchDeerRgKYE054QutGDnvRlIIMYwzgmMYv5LGAJS1nNGnayh30c"
    "4C9O8jePeMw7Ysd3xkhKclKSmSzkpjBB1KAmdWjAt/yP1rQjhK70ZSDDGcU4JjCZGSxmKb/yG+vYwja2s5sIDnKck5zmLLe5"
    "SxQPieY5Mbzni88DAlKRgyDKU5GvaUpr2tCJELrRh2GMYSyzmctilrCKX9nCTvawj4NcIoponhPDRwISuB/4nIQkJhmpSUM6"
    "MpGFbOQmP4UpRyUqU4PafE1DWtKK1vSmL+OYQijzmM8CFhLGataxlQgOcog/OcE5LnCJG0QRQ+yEOpj4pCA1aclLEYpTjprU"
    "ozmtaEN7fqA7PejDYIYymjGMYxJTmM4MQpnHYpazktVsZBN/sJ2d7OMwx7jOfV4Rw1tiJdINfEo8EpCClKQnI0UoShmqUo+G"
    "NOE7WtCOToxgKqEsIIxl/MZ2dhPBAQ5xjBNc5Dq3uc9DonlODB/57AsdQVKSk5LUZCAHOSlMGYKoQS3q0YI2dKAvAxjEOKYw"
    "m7nMZwXr2cJejnCGs9zgNg+J5i0BX5qN5KQgFWnJQD7yU4giFKcEpalARSpTmzq0pBU/0IlujGE8MwlnPRvYxHZ2EMEB/uQS"
    "V7jBbaJ5wlNe8Uliz5x4xCcZKUhPVrKTg3zkpwAFKUQJShJEFerTmG9pSjNa0Jp2hNCNwYxgItOYRSjzCGMpywnnN9aznUMc"
    "5RTnuUcMAUn0GElJR3YKUJjilCSIilSiFrVpTAid6UpPetOXoYxgFBOYxM9MZxazmcd8lrCS1YSznh3sYz/HOM8FrnCdOzzg"
    "MU94QQxviZVUj5CQRHxBMtKRgUDykp/iVKcuX9OUlrSiLe1oT1d6MpAhDGM4o5nKdGYwk3ksYAlrWMvvbOQP9nGey1zhAVFE"
    "85J3BCSTKVKTngxkJCs5qcHXfEcLWtKeDnRmKMMYzc/MYg6LWEY4m4jgX25zh/tE8ZRnvOA9cXwYfkJ8viQTWchPAYpQmiAa"
    "8A3/owUd6URX+tCP/gxhMjOYxWrWsJFN7GQ3B4nkBtE84eN/H6op3BnE4RMSkJxc5KUAhShOKapQjRrUpC71aUZrfiSELoxj"
    "EvMJZx1/sIXt7OYE57jAv1wkipd84NOUZiIVqUlDVnKQi7zkpwjFKE416vI1TfiWYNrSiR70ZQDTWcB6NrCZLWxnH0c4xVXu"
    "8pRXvCGGWKnsjXikICXpyEw2ClOMEpSlIrVpTjA/EkIXutGbYYxkKtOYzgxCCecPdrCTvRzlb05zjgvc4gFPeU3c1HJHAhLy"
    "JWlJR2GCqEsj/kdz2tOJLvSgHwMYyk+MYSKhzGUBiwhjFWv5jY1sYhe7OcUtbvOYpzzjFe/5SLw0KpWkpCYtmclOIAUpQnHK"
    "UoEqVKcGtWjGd7SkDcG0pyM96Et/BjGYCfzCTEIJYxkrWM0afmM9W9nJbiI4zmVucJeHRPOEF7wldlrnjEQkIw1ZyEtxSlGN"
    "ejSmGc1pSWs6MJ6JTCOUcH5lK7vYwz4iOMyf/MVxTvEPV7nDXR7ymCc84yUxfCBuOu8RxCcRSUlGJgIpRBFK8hUNaEgrOtKZ"
    "fgxgEBOZzVyWsoLf2MhmdrKLvzjJKc5ygatc5ya3uM8DnvCaNOm91xJERSpTharU5ivq8S3f057BDGUsk5jBbJaxkl/ZwB/s"
    "5SCRXOUaD3nOC17yihg+EjeDs0YSkpOSVASSm7zkoyDlqExN6lCfBnzDj3SgIyH0ZiBDGMFIfuYXQgljCctYTwSHOMkpLnCJ"
    "W9zhAY+J5g0fiJVRTklKclKQnszkIg9FKEY5KlCN2tSlMS0Ipi0hdKEfgxjGcEYxlRnMJoylrGQLu9nDPvZziEiOcoKzXOMu"
    "93lH7Ez6mAQkIRk5yUsxSlCOCtSjBUMYyQQmM4M5LGQJS1nOSlaxjk1sZS+HOccFLvKAKJ7wkle85dPMzhOJSEJK0pCRzGQj"
    "OzkIJDelKUsFalGfJjTne4LpQHf60p/RjGMKM1nDWjayje0c5Szn+ZcrXOMhz3jz36xZ3BUkIQXpyEABilGcEpSiNGUoR3mq"
    "UJU61KcBIXSnJ30ZzghGM5ZxTGYxYaxiNetYzx/sYS+RHOMvzvCE17zjPTmy2h95KUhRilOJ6tTnW1rQinZ0pBNd6M8QhjKZ"
    "n/mF+SxgIVvYwS4iOMxJHvCY57zl82zyyJekIxNZCaQQpShDWYKoQCUqU4WqVKcOTWjK/2hOC9oQTFv60o/+DGI4oxnLOCYw"
    "nXksZCmrWMsuDnKII5ziLOe4xV3ucZ8HPOQRz3jBe9Jkd1eTkWzkID/FKUFJSlOJmtTmO74nhJ704yfGMYmpzGI+C1jEMlYS"
    "zq9sYhs72c1BIjnKP5zjIle5xT0e8oQXvOcDcXLobuKTmAxkIpB8FKUsFalEZapQl/o0oCGNaEJLgmnLD3SnD/0ZwFBGM54J"
    "TOYXQpnHYjaxm7+4yCVuEcUT3vCeD3yW0+x8TlJSkoGMZCIXgZSjInVoSSva0YEQutCNnvSmP0MYxhgmMIn5rGEbBzjEP1zk"
    "Ele4yi1e84a3vOcjCXPpaFKThvRkJRs5yU0+ClOEMlShKnWpRwO+piHNaEkrgvmBrvRmIMMYyTjGM4FJzGIOc1nIKn5lI9vZ"
    "xW72c4QTXOY293hANE94R+JAv4lM5KEEZShPJWpQi3r8j+YE051e9KEfQxjGSEYzlZnMIpQ5zGMBi1jBGtazjT0cIJIjHOUE"
    "pznHFa5yjevcJpqnPOcVH4mXW65IRArSko/8FKQolahGDZrRgjZ0pweDGEooC1nOClazl/0c4CCRHOU4JzjFGc5zgctc5yGf"
    "5HG38iXpyUBGslGQ4pSgNOWpRh0a0JAmtKcrfRjAIAYzhon8zDRmM4+FLGMF4aznD7awlR3s4xDHOcUVbhBFnLx6hngkIDkp"
    "SEsOClCEEpSnEtWoTWO+pRnB/EAHujCQoYxgDJOZzgwWEMYKVrGdHUSwn2Oc5BRnuMw1bnCP+zznFW95xwdi5fPeRTzik5jU"
    "ZCYngeQjiIrUpC4NaMi3tKIbAxjEEMYxgclMYS6LCWMFO9nHcU5wktNc4F+ucZPb3OMxT4nhLXHyyw6fkZBEpCA7echLPgpT"
    "lBKUpgwVqEINvqIe9WlMC1rRmrZ0owd9GckkpjKdmcxiLosJZwNbOchhIjnOac5wiftE8ZBHPCaa57ziDbELeB4k5AuSkYL0"
    "5CCQAhSlEnWpR2OaEEJ3ejGIoYxkDLOYRzjr2MgWthPJOa4TzUte8ZbkBWWc9GQkM4UoQmnKUJYqVOUrmtGKYDoSQmeGMYJJ"
    "TGMWcwhjOfs5ynH+4TJXuM49nhOrkAzwKfFISBJSkpaM5KM0ZShHZerSjJa0Ipi2tKcb/RnBT0xnLotYzBKW8yu/8wfb2M5u"
    "jnCUE5ziDOc4zz0e8oznvCBuYXOTiFzkpjBFKEMQ9WnINzSiGcH8wI8MYjBDGMoYxjOZaYQyn8WsZBWrWcMuDnCYSI5yirNc"
    "4Bp3uMcj3vCOOEWcPRKQmFRkJzf5KUAhilKW2jSiBe3oxBCGM5LRjGM6vzCTecxnIWEsYzlr2cwuIjjCUf7mNJe5w2PeEMNb"
    "3vOBT4uamyRkJAs5CKQwJShDWYKoQCUqU5M61KMRvRnCSEYzlonMZg5zmcditrCdSM5yjmtc5w53ecxzYnjLB2IVs2/ikYjE"
    "JCczeShFZarxHa3oSk/6MpM5rGQru9nH35zmLOe4zH1e8JrPissmX5CCjGQnJ7kpQgVq8DUtacUAhvMTo5jIL4SxlHDW8zub"
    "2coRjnKMf7jHQx7zjBe8IYZ4JcxFSrKSnRzkpjBFKEUDWvEjwxjOCEYykUnMYxFhhLOOTexmDxEcJJLjnOQfrnKTKB7ymBje"
    "Equk7iYBCfmCVGQgMwUpShnKUpkqVKcOjWlJG37gRzrSiR70ZxST+IVQ5hDGEpaxkh3sYjdHOcZfHOdvTnOGf7jDPR4Rq5Rn"
    "TzLSkYlsZCeQPBSmNGUJogI1qE8DGtGcVrShHR3oQS/6MpChTGYavzCLRaxlPRvZz0GOcJwTnOUct7jDXaJ4xjs+ELe07yiS"
    "kYp0ZCYbuShMNerRkEY0oSWtaUsv+jOUEUxkGstZSzjr2cw2DnCQPznCSU5zg5vc4SGPeMIznvOGuGXsnYQk4kvSk4Vs5CM/"
    "JahKTZrTgu9pRQid6ck4JjGZnwllNksIZye7iOAAkRzlLP9yh+d8+G+usjLN5yQgKclJRWrSk4GM5CI/pShLVapRnRrU4xsa"
    "0ZimfEcw7QhhBCMZwwx2sJf9HORPTnCaM5zjX65zm/s85jlxyjmDpCUDGclELnKTh4IUpxzlqUBFqlCbJjSjFe3oTi96M4Sh"
    "DGcck5jDan5jEzvYzR4OcohznOdfLnGZu9zjGc95V+7//2j6CXFJRFLSkZ5s5CSQYpQgiErUogHN6UQIXRnKOCYwmwUsZBFL"
    "WcZafmczeznAIY5ylnP8yxXuco9HRPOUt7znI3HKm5lUpCUdgeQlH0WpRn2a0po2tKUd3ehDfwYyjFHMYg7zWMJSVrKa9Wxg"
    "GzvZxV4O8Cd3ecoz3hBDnAreDUhMMpKTk1zkoTDFKEEpGhFMW9rTlUH8xEhmMpsFLGQxq1jLRjaxkwj2c5gTnOMil7nNI6J5"
    "y2cV9QJJSEkg+alETerQlJ70oS8jmchkVrKdHewjguOc5CwXuc1dklVytklDOjKSl+KUoDwVqE9D2tCJrgxmGDMJZT4LWMgi"
    "wljKClbxO1vYzh4Oc4S/uMJ1bnGPaF7ylk8r6yNSkZr0ZCOQ3JSgHEFUpS71aMg3NKIxzehARzrRle4MYCBD+JkpzGQBC1nB"
    "r6xnO7vZy2FOcJIr3OQuz3hNDG95R5wq+onEJCMlWchBYYpTm4Z0YzijGMvPzGAO85jPAsJYzjo2sYWd7CeSM1zgGnd5yBti"
    "eMd7YlXV9yQiMalJQzpykp/ClKUiNWhBS1rTjhC60o0e9GIAgxjPRCYzjVDmMJcFrGMDW9jFbvYRwX6OcJZ/ucJNbnOXh7wk"
    "hvfErmbPxCcd6clARnISSDGCqE4t6lKfRjSmGc3pSA9GM5bJTGUas1lIGEtYRjgRHOIYJ/ibs1ziOs/+m6+6LJOI5KQgFVkJ"
    "JC+FKENFqvENjWhMezrTjZ70YhijmcBEpjKNucxjDTvZy34Oc4JL3OIOD3nN5zXMRSaykJ3c5KMA5alAVerQkBYE047O9GA4"
    "k/iFUOYTxnJ2sIfDHOcEpzjHDW5ym3s84TVviFXTHUlcPiM+X5KSbBSiGFWpxVd8QyOa8C3NacH39GQAQxjFeCYwiSlMZyGL"
    "WcJqwtnEfv7iJJe5zh0e84aPxKtlRhKRnMxkJQ/5yE8BSlKGylShLo1ozLe0JoQ+DGAgY5nANGYwh7ksYhkrWMlvbGQHxzjB"
    "Kc5wjiie8owXxKstnyQlGcnJQDbyUJKylKMSVajFV9SnCd/xPa0Jph0/0InOdKcnvRjEYEYxjbnMZzFLWEY46znIP1wmisdE"
    "84L4dWSXjGQmByWpQGWqUZP6NOM7WtOWdnSgC93oST8GMJDBDOEnxvIz01nJFvYQwSEiOcU1nhC7rqySkNRkIAvZyE4e8pKP"
    "gpSiGvVpRFOa051+jGQs45nELBYQxlJWsIYN7OcwRzhBFE94yjs+8sVX+p7M5CEfBSlEUUpQnjp8Q1Pa0oPe9KE/45nARCYz"
    "hemEMpe17CCCw0RymjNc5Bq3eUusevZGUlKSlv8DLoGyLQ=="
)
CAND = np.frombuffer(zlib.decompress(base64.b64decode(CAND_B64)), np.int32)
print(f'{len(CAND)} rotation candidates')


class Imgs(Dataset):
    """Resized directly to a square. Aspect-preserving letterboxing was tested
    and scored worse: the padding bands are identical across images, and the
    whitening then treats that shared structure as signal."""

    def __init__(self, paths, rotate=0):
        self.paths, self.rotate = paths, rotate
        self.tf = transforms.Compose([
            transforms.Resize((SIZE, SIZE),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            im = Image.open(self.paths[i]).convert('RGB')
            if self.rotate:
                im = im.rotate(self.rotate, expand=True)
            return self.tf(im), i
        except Exception as e:
            print('!', self.paths[i].name, e, flush=True)
            return torch.zeros(3, SIZE, SIZE), i


def gem(pt, p=3.0, eps=1e-6):
    """Generalised mean over patch tokens. p=3 sits between average pooling
    (p=1) and max (p to infinity); concatenated with CLS it beat either alone."""
    return pt.clamp(min=eps).pow(p).mean(1).pow(1 / p)


def extract(model_name, subset, rotate, tag, batch=4):
    """CLS + GeM features for `subset`, checkpointed every 3 minutes.

    Three multi-hour jobs were lost before checkpointing was added; the fourth
    survived a segfault at 56 percent. Re-running this cell resumes.
    """
    ck, dn = WORK / (tag + '_ckpt.npy'), WORK / (tag + '_done.npy')
    model = torch.hub.load('facebookresearch/dinov2', model_name,
                           verbose=False).eval().cuda().half()
    dim = 3072 if 'vitg' in model_name else 2048
    feats = np.load(ck) if ck.exists() else np.zeros((len(paths), dim), np.float32)
    done = np.load(dn) if dn.exists() else np.zeros(len(paths), bool)
    todo = np.array([i for i in subset if not done[i]])
    print(f'{tag}: {len(todo)} to embed, {int(done.sum())} already done')
    if len(todo):
        dl = DataLoader(Imgs([paths[i] for i in todo], rotate), batch_size=batch,
                        num_workers=4, pin_memory=True)
        t0 = last = time.time()
        with torch.no_grad():
            for x, sl in dl:
                f = model.forward_features(x.cuda(non_blocking=True).half())
                v = torch.cat([f['x_norm_clstoken'], gem(f['x_norm_patchtokens'])], 1)
                idx = todo[sl.numpy()]
                feats[idx] = v.float().cpu().numpy()
                done[idx] = True
                if time.time() - last > 180:
                    np.save(ck, feats); np.save(dn, done); last = time.time()
                    n = int(done[subset].sum())
                    r = max(n / (time.time() - t0), 1e-6)
                    print(f'  {n}/{len(subset)}  ETA {(len(subset)-n)/r/60:.0f} min',
                          flush=True)
        np.save(ck, feats); np.save(dn, done)
    del model
    torch.cuda.empty_cache()
    return feats


In [ ]:
ALL = np.arange(len(paths))
G    = extract('dinov2_vitg14', ALL,  0,   'g518')      # ~1.0 h
L0   = extract('dinov2_vitl14', ALL,  0,   'l518')      # ~0.5 h
L180 = extract('dinov2_vitl14', CAND, 180, 'l518r180')  # ~5 min, candidates only
np.save(WORK / 'features_g.npy', G)
np.save(WORK / 'features_l518.npy', L0)
np.save(WORK / 'features_l518_rot180.npy', L180)
print('extraction complete', G.shape, L0.shape, L180.shape)


In [ ]:
l2 = lambda a, eps=1e-12: a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)


def whiten(x, dim, alpha=1.0):
    mu = x.mean(0, keepdims=True)
    _, s, vt = np.linalg.svd(x - mu, full_matrices=False)
    sc = (s[:dim] / np.sqrt(len(x) - 1)) ** alpha + 1e-8
    return l2((x - mu) @ vt[:dim].T / sc).astype(np.float32)


g  = l2(G.astype(np.float64))
Lu = l2(L0.astype(np.float64))
Lr = l2(L180.astype(np.float64))

# ONE transform, fitted on the upright corpus and applied to both views.
# Whitening each view with its own SVD puts them in different spaces, the
# similarities stop being comparable, and it silently flips nothing.
mu = Lu.mean(0, keepdims=True)
_, sv, vt = np.linalg.svd(Lu - mu, full_matrices=False)
sc = sv[:1536] / np.sqrt(len(Lu) - 1) + 1e-8
proj = lambda z: l2((z - mu) @ vt[:1536].T / sc).astype(np.float32)
W, W180 = proj(Lu), proj(Lr)


def top1(Q, ex):
    """Best similarity against the upright corpus, excluding self."""
    best = np.zeros(len(Q), np.float32)
    for i in range(0, len(Q), 256):
        S = Q[i:i+256] @ W.T
        for r in range(len(S)):
            S[r, ex[i+r]] = -1
        best[i:i+256] = S.max(1)
    return best


gain = top1(W180[CAND], CAND) - top1(W[CAND], CAND)
flip = CAND[gain > MARGIN]
Lc = Lu.copy()
Lc[flip] = Lr[flip]
print(f'flipped {len(flip)} of {len(CAND)} candidates at margin {MARGIN}')

# per-block L2 so neither block dominates, then FULL-width whitening
x = np.hstack([l2(g), l2(Lc)])
print('concatenated', x.shape)
f = whiten(x, x.shape[1], 1.0)

import pandas as pd
df = pd.DataFrame(f, columns=[f'feature_{i}' for i in range(f.shape[1])])
df.insert(0, 'image_name', list(names))
df['ID'] = df['image_name']
df.to_csv(WORK / 'submission.csv', index=False, float_format='%.6f')
print(f'wrote submission.csv: {len(df)} rows x {df.shape[1]} cols  (expect LB 0.86912)')
